In [1]:
!pip -q install datasets segmentation-models-pytorch

In [2]:
from datasets import load_dataset
import numpy as np

ds = load_dataset("hassanjbara/AI4MARS")
print(ds)


README.md:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

data/train-00000-of-00013.parquet: reconstructing file:   0%|          |  0.00B /  475MB            

data/train-00000-of-00013.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00013.parquet: reconstructing file:   0%|          |  0.00B /  476MB            

data/train-00001-of-00013.parquet: downloading bytes:           |  0.00B            

data/train-00002-of-00013.parquet: reconstructing file:   0%|          |  0.00B /  471MB            

... (28 outputs left out)

Generating test_min1 split:   0%|          | 0/322 [00:00<?, ? examples/s]

Generating test_min2 split:   0%|          | 0/322 [00:00<?, ? examples/s]

Generating test_min3 split:   0%|          | 0/322 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label_mask', 'rover_mask', 'range_mask', 'has_masks', 'has_labels'],
        num_rows: 18130
    })
    test_min1: Dataset({
        features: ['image', 'label_mask', 'rover_mask', 'range_mask', 'has_masks', 'has_labels'],
        num_rows: 322
    })
    test_min2: Dataset({
        features: ['image', 'label_mask', 'rover_mask', 'range_mask', 'has_masks', 'has_labels'],
        num_rows: 322
    })
    test_min3: Dataset({
        features: ['image', 'label_mask', 'rover_mask', 'range_mask', 'has_masks', 'has_labels'],
        num_rows: 322
    })
})


In [3]:
import requests, pandas as pd, json

wp = requests.get("https://mars.nasa.gov/mmgis-maps/MSL/Layers/json/MSL_waypoints.json").json()
df = pd.DataFrame([f["properties"] for f in wp["features"]])
box = df[(df.sol >= 4534) & (df.sol <= 4805)].copy()
print(len(box), "stops")

fake = [{"sol": int(r.sol), "lon": float(r.lon), "lat": float(r.lat),
         "soil_pct": 0.5, "bedrock_pct": 0.3, "sand_pct": 0.15, "bigrock_pct": 0.05,
         "image": "", "note": "FAKE numbers for building the map",
         "type": "boxwork" if r.sol >= 4600 else "stop"} for r in box.itertuples()]
json.dump(fake, open("stops_fake.json", "w"), indent=1)
box.to_csv("boxwork_waypoints.csv", index=False)
print("saved ✅")

65 stops
saved ✅


In [4]:
import matplotlib.pyplot as plt

print("has_labels:", sum(ds["train"]["has_labels"]), "/", len(ds["train"]))
print("has_masks :", sum(ds["train"]["has_masks"]),  "/", len(ds["train"]))

i = next(k for k in range(200) if ds["train"][k]["has_labels"] and ds["train"][k]["has_masks"])
row = ds["train"][i]
for k in ["image", "label_mask", "rover_mask", "range_mask"]:
    a = np.array(row[k]); print(k, a.shape, a.dtype, np.unique(a)[:10])

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, k in zip(ax, ["image", "label_mask", "rover_mask", "range_mask"]):
    x = np.array(row[k]); x = x[..., 0] if x.ndim == 3 and k != "image" else x
    a.imshow(x, cmap="gray" if k == "image" else "viridis"); a.set_title(k); a.axis("off")
plt.show()

has_labels: 16064 / 18130
has_masks : 18096 / 18130
image (1024, 1024) uint8 [ 1  2  3  4  5  6  7  8  9 10]
label_mask (1024, 1024) uint8 [  0   1   3 255]
rover_mask (1024, 1024) uint8 [0 1]
range_mask (1024, 1024) uint8 [0]


<Figure size 1600x400 with 4 Axes>

In [5]:
for k in range(20):
    r = ds["train"][k]
    if not (r["has_labels"] and r["has_masks"]): continue
    lab = np.array(r["label_mask"]); rov = np.array(r["rover_mask"]); rng = np.array(r["range_mask"])
    far = rng > 0
    print(k, "rover%", round((rov > 0).mean()*100, 1),
          "| range=1 %", round(far.mean()*100, 1),
          "| of range=1 pixels, unlabeled(255) %", round((lab[far] == 255).mean()*100, 1) if far.any() else "-")

1 rover% 46.6 | range=1 % 0.0 | of range=1 pixels, unlabeled(255) % -
2 rover% 5.6 | range=1 % 30.6 | of range=1 pixels, unlabeled(255) % 100.0
3 rover% 50.6 | range=1 % 27.2 | of range=1 pixels, unlabeled(255) % 100.0
4 rover% 0.0 | range=1 % 33.6 | of range=1 pixels, unlabeled(255) % 100.0
5 rover% 10.3 | range=1 % 0.0 | of range=1 pixels, unlabeled(255) % -
6 rover% 9.1 | range=1 % 0.0 | of range=1 pixels, unlabeled(255) % -
7 rover% 0.0 | range=1 % 13.0 | of range=1 pixels, unlabeled(255) % 100.0
9 rover% 62.5 | range=1 % 0.0 | of range=1 pixels, unlabeled(255) % -
10 rover% 59.6 | range=1 % 0.0 | of range=1 pixels, unlabeled(255) % -
12 rover% 0.0 | range=1 % 0.0 | of range=1 pixels, unlabeled(255) % -
14 rover% 0.0 | range=1 % 23.7 | of range=1 pixels, unlabeled(255) % 100.0
15 rover% 46.5 | range=1 % 0.0 | of range=1 pixels, unlabeled(255) % -
16 rover% 59.7 | range=1 % 0.0 | of range=1 pixels, unlabeled(255) % -
17 rover% 0.0 | range=1 % 14.6 | of range=1 pixels, unlabeled(255)

In [6]:
for k in [1, 3, 9, 10, 15, 16]:
    r = ds["train"][k]
    lab = np.array(r["label_mask"]); rov = np.array(r["rover_mask"]) > 0
    print(k, "rover% ", round(rov.mean()*100, 1),
          "| of rover=1 pixels, unlabeled(255) %", round((lab[rov] == 255).mean()*100, 1))


1 rover%  46.6 | of rover=1 pixels, unlabeled(255) % 100.0
3 rover%  50.6 | of rover=1 pixels, unlabeled(255) % 100.0
9 rover%  62.5 | of rover=1 pixels, unlabeled(255) % 100.0
10 rover%  59.6 | of rover=1 pixels, unlabeled(255) % 100.0
15 rover%  46.5 | of rover=1 pixels, unlabeled(255) % 100.0
16 rover%  59.7 | of rover=1 pixels, unlabeled(255) % 100.0


Setup (Drive + prepare)

In [8]:
from google.colab import drive; drive.mount("/content/drive")
import os, torch, numpy as np, segmentation_models_pytorch as smp, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
os.makedirs("/content/drive/MyDrive/firmground", exist_ok=True)
IMG, LAB, ROV, RNG = "image", "label_mask", "rover_mask", "range_mask"
SIZE, dev = 512, "cuda"
MEAN = torch.tensor([.485,.456,.406])[:,None,None]; STD = torch.tensor([.229,.224,.225])[:,None,None]

def one(x):
    a = np.array(x); return a[..., 0] if a.ndim == 3 else a
def prepare(row):
    img = np.array(row[IMG].convert("RGB"))
    lab = one(row[LAB]).astype(np.int64).copy()
    for k in (ROV, RNG):
        if row.get(k) is not None:
            lab[one(row[k]) > 0] = 255          # rover + far (>30 m) → ignore
    return img, lab

class Mars(Dataset):
    def __init__(s, hf, aug=False): s.hf, s.aug = hf, aug
    def __len__(s): return len(s.hf)
    def __getitem__(s, i):
        img, lab = prepare(s.hf[i])
        x = torch.from_numpy(img).permute(2,0,1).float() / 255
        x = F.interpolate(x[None], size=(SIZE,SIZE), mode="bilinear", align_corners=False)[0]
        y = torch.from_numpy(lab)[None,None].float()
        y = F.interpolate(y, size=(SIZE,SIZE), mode="nearest")[0,0].long()
        if s.aug:
            if torch.rand(1) < .5: x, y = x.flip(-1), y.flip(-1)
            x = (x * (0.8 + 0.4*torch.rand(1))).clamp(0, 1)
        return (x - MEAN) / STD, y
print("A ok")

Mounted at /content/drive
A ok


we remove el tsawer eli blech labels + validation


In [9]:
labeled = ds["train"].filter(lambda h: h, input_columns=["has_labels"])
print("labeled:", len(labeled))                 # testanna ~16064
sp = labeled.train_test_split(test_size=0.1, seed=42)
train_hf, val_hf = sp["train"], sp["test"]
print("train:", len(train_hf), "| val:", len(val_hf))

Filter:   0%|          | 0/18130 [00:00<?, ? examples/s]

labeled: 16064
train: 14457 | val: 1607


Class weights

In [10]:
counts = np.zeros(4)
for i in np.random.RandomState(0).choice(len(train_hf), 2000, replace=False):
    _, lab = prepare(train_hf[int(i)]); v = lab[lab != 255]
    counts += np.bincount(v, minlength=4)[:4]
share = counts / counts.sum(); weights = 1 - share
print("share soil/bedrock/sand/bigrock:", share.round(3))
print("weights:", weights.round(3))

share soil/bedrock/sand/bigrock: [0.36  0.508 0.121 0.011]
weights: [0.64  0.492 0.879 0.989]


TRAINING

In [11]:
model = smp.DeepLabV3Plus("resnet50", encoder_weights="imagenet", classes=4).to(dev)
loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=dev), ignore_index=255)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.cuda.amp.GradScaler()
dl = DataLoader(Mars(train_hf, aug=True), batch_size=8, shuffle=True, num_workers=2, drop_last=True)
for epoch in range(4):
    model.train(); tot = 0
    for step, (x, y) in enumerate(dl):
        x, y = x.to(dev), y.to(dev); opt.zero_grad()
        with torch.autocast("cuda", dtype=torch.float16):
            loss = loss_fn(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tot += loss.item()
        if step % 100 == 0: print(epoch, step, round(loss.item(), 3))
    torch.save(model.state_dict(), f"/content/drive/MyDrive/firmground/mars_e{epoch}.pt")
    print("epoch", epoch, "mean loss", round(tot/(step+1), 4), "→ saved ")

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

/tmp/ipykernel_7566/1156745647.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


0 0 1.489
0 100 0.341
0 200 0.329
0 300 0.383
0 400 0.122
0 500 0.223
0 600 0.431
0 700 0.144
0 800 0.636
0 900 0.255
0 1000 0.391
0 1100 0.227
0 1200 0.165
0 1300 0.124
0 1400 0.042
0 1500 0.247
0 1600 0.464
0 1700 0.138
0 1800 0.184
epoch 0 mean loss 0.2572 → saved 
1 0 0.079
1 100 0.5
1 200 0.048
1 300 0.15
1 400 0.143
1 500 0.157
1 600 0.648
1 700 0.076
1 800 0.269
1 900 0.136
1 1000 0.061
1 1100 0.169
1 1200 0.046
1 1300 0.198
1 1400 0.063
1 1500 0.236
1 1600 0.203
1 1700 0.1
1 1800 0.049
epoch 1 mean loss 0.1631 → saved 
2 0 0.102
2 100 0.133
2 200 0.097
2 300 0.197
2 400 0.197
2 500 0.079
2 600 0.072
2 700 0.063
2 800 0.048
2 900 0.09
2 1000 0.099
2 1100 0.049
2 1200 0.124
2 1300 0.191
2 1400 0.45
2 1500 0.138
2 1600 0.122
2 1700 0.133
2 1800 0.12
epoch 2 mean loss 0.1415 → saved 
3 0 0.213
3 100 0.13
3 200 0.095
3 300 0.121
3 400 0.189
3 500 0.089
3 600 0.056
3 700 0.219
3 800 0.005
3 900 0.191
3 1000 0.048
3 1100 0.222
3 1200 0.146
3 1300 0.182
3 1400 0.068
3 1500 0.033
3 1600

In [12]:
import json
def evaluate(model, hf):
    cm = np.zeros((4, 4), np.int64)
    model.eval()
    with torch.no_grad():
        for x, y in DataLoader(Mars(hf), batch_size=8, num_workers=2):
            p = model(x.to(dev)).argmax(1).cpu().numpy(); y = y.numpy(); m = y != 255
            cm += np.bincount(4*y[m] + p[m], minlength=16).reshape(4, 4)
    iou = np.diag(cm) / (cm.sum(0) + cm.sum(1) - np.diag(cm) + 1e-9)
    base = cm.sum(1) / cm.sum()                       # all-soil baseline: soil IoU = soil share
    return {"iou": iou.round(3).tolist(), "miou": round(float(iou.mean()), 3),
            "pixel_acc": round(float(np.diag(cm).sum() / cm.sum()), 3),
            "baseline_soil_iou": round(float(base[0]), 3), "baseline_miou": round(float(base[0] / 4), 3),
            "cm": cm.tolist()}

res = {}
for s in ["test_min3", "test_min1"]:
    res[s] = evaluate(model, ds[s])
    r = res[s]
    print(s, "| IoU soil/bedrock/sand/bigrock:", r["iou"], "| mIoU", r["miou"], "| pixel acc", r["pixel_acc"],
          "| baseline mIoU", r["baseline_miou"])
json.dump(res, open("/content/drive/MyDrive/firmground/metrics.json", "w"))
print("saved metrics.json")

test_min3 | IoU soil/bedrock/sand/bigrock: [0.981, 0.941, 0.966, 0.394] | mIoU 0.821 | pixel acc 0.983 | baseline mIoU 0.128
test_min1 | IoU soil/bedrock/sand/bigrock: [0.921, 0.826, 0.864, 0.12] | mIoU 0.683 | pixel acc 0.931 | baseline mIoU 0.115
saved metrics.json


In [13]:
import requests, os, time, pandas as pd
API = "https://mars.nasa.gov/api/v1/raw_image_items/"
def navcam(sol, cam):
    q = {"order": "sol desc", "per_page": 100, "page": 0, "condition_1": "msl:mission",
         "condition_2": f"{sol}:sol:in", "condition_3": f"{cam}:instrument"}
    items = requests.get(API, params=q, timeout=30).json().get("items", [])
    return [i for i in items if not i["is_thumbnail"]
            and (i.get("extended") or {}).get("sample_type") == "full"]

os.makedirs("navcam", exist_ok=True); rows = []
for sol in range(4600, 4620):
    for cam in ("NAV_RIGHT_B", "NAV_LEFT_B"):
        try:
            for it in navcam(sol, cam):
                fn = f"navcam/{it['imageid']}.jpg"
                if not os.path.exists(fn):
                    open(fn, "wb").write(requests.get(it["https_url"], timeout=60).content)
                rows.append({"file": fn, "sol": it["sol"], "site": it["site"], "drive": it["drive"]})
        except Exception as e:
            print(sol, cam, "error:", e)
        time.sleep(0.5)
photos = pd.DataFrame(rows); print(len(photos), "photos")

460 photos


In [14]:
from PIL import Image
# 1) keep only full 1024x1024 frames (drop small sub-frames)
photos["size"] = [Image.open(f).size for f in photos.file]
full = photos[photos["size"] == (1024, 1024)].copy()
print("full frames:", len(full), "/", len(photos))

# 2) join each photo to Curiosity's position (site + drive)
geo = full.merge(df[["site", "drive", "lon", "lat"]], on=["site", "drive"], how="left")
print("with exact waypoint:", geo.lon.notna().sum(), "| without:", geo.lon.isna().sum())
geo.to_csv("navcam_geo.csv", index=False)


full frames: 460 / 460
with exact waypoint: 425 | without: 35


In [15]:
import matplotlib.pyplot as plt
sub = geo.dropna(subset=["lon"]).sample(24, random_state=0)
fig, ax = plt.subplots(4, 6, figsize=(18, 12))
for a, (_, r) in zip(ax.flat, sub.iterrows()):
    a.imshow(Image.open(r.file), cmap="gray"); a.set_title(f"sol {r.sol}", fontsize=9); a.axis("off")
plt.tight_layout(); plt.show()

<Figure size 1800x1200 with 24 Axes>

we remove the sky


In [16]:
import numpy as np
geo = geo.dropna(subset=["lon"]).reset_index(drop=True)
def texture(f):
    a = np.array(Image.open(f).convert("L"), float)
    return a[int(0.4*a.shape[0]):].std()          # how much detail in the lower part
geo["texture"] = [texture(f) for f in geo.file]
cand = geo[geo.texture > 20].reset_index(drop=True)  # sky / blurry = almost no detail
print("kept:", len(cand), "/", len(geo))
print(cand.groupby(["site", "drive"]).size())     # photos per stop

kept: 413 / 425
site  drive
117   2166      2
      2370     40
      2490     58
      2538     59
      2730     59
      2850     59
118   0        55
      162      42
      198      39
dtype: int64


In [17]:
for (site, drive), g in cand.groupby(["site", "drive"]):
    g = g.head(12)
    fig, ax = plt.subplots(2, 6, figsize=(18, 6))
    for a in ax.flat: a.axis("off")
    for a, (i, r) in zip(ax.flat, g.iterrows()):
        a.imshow(Image.open(r.file), cmap="gray"); a.set_title(f"#{i} · sol {r.sol}", fontsize=10)
    fig.suptitle(f"stop site {site} drive {drive}"); plt.tight_layout(); plt.show()


<Figure size 1800x600 with 12 Axes>

<Figure size 1800x600 with 12 Axes>

<Figure size 1800x600 with 12 Axes>

<Figure size 1800x600 with 12 Axes>

<Figure size 1800x600 with 12 Axes>

<Figure size 1800x600 with 12 Axes>

<Figure size 1800x600 with 12 Axes>

<Figure size 1800x600 with 12 Axes>

<Figure size 1800x600 with 12 Axes>

nekhtarou 3 tsawer

In [18]:
def pick_score(f):
    a = np.array(Image.open(f).convert("L"), float); h = a.shape[0]
    top  = a[: int(0.2*h)]                    # ground everywhere → top is textured too (no sky)
    band = a[int(0.4*h): int(0.85*h)]         # the part the AI will read
    no_sky   = min(top.std() / 25, 1.0)        # sky = flat → low
    detail   = min(band.std() / 30, 1.0)       # ground texture
    dark     = (band < 35).mean()              # rover wheels / deck / deep shadow are dark
    return no_sky + detail - 3 * dark

cand["score"] = [pick_score(f) for f in cand.file]
picked = (cand.sort_values("score", ascending=False)
              .groupby(["site", "drive"]).head(3)
              .sort_values(["site", "drive"]).reset_index(drop=True))
print(len(picked), "photos picked for", picked.groupby(["site","drive"]).ngroups, "stops")


26 photos picked for 9 stops


In [19]:
fig, ax = plt.subplots(len(picked)//3 + 1, 3, figsize=(12, 4*(len(picked)//3 + 1)))
for a in ax.flat: a.axis("off")
for a, (i, r) in zip(ax.flat, picked.iterrows()):
    img = np.array(Image.open(r.file).convert("L")); h = img.shape[0]
    a.imshow(img, cmap="gray")
    a.axhline(0.4*h, color="yellow"); a.axhline(0.85*h, color="yellow")   # the zone the AI reads
    a.set_title(f"#{i} · site {r.site} drive {r.drive} · sol {r.sol}", fontsize=9)
plt.tight_layout(); plt.show()

<Figure size 1200x3600 with 27 Axes>

In [20]:
import json, os
from pathlib import Path
keep = [2,3,4,5,6,7,8,10,12,13,14,15,17,18,19,20,21,22,23,24]
sel = picked.loc[keep].reset_index(drop=True)

COL = np.array([[150,110,60],[90,90,90],[240,200,60],[200,40,40]])   # soil, bedrock, sand, big rock
def predict(path):
    img = np.array(Image.open(path).convert("RGB")); H, W = img.shape[:2]
    x = torch.from_numpy(img).permute(2,0,1).float() / 255
    x = F.interpolate(x[None], size=(SIZE,SIZE), mode="bilinear", align_corners=False)[0]
    with torch.no_grad():
        p = model(((x - MEAN) / STD)[None].to(dev)).argmax(1)[0].cpu().numpy().astype(np.uint8)
    return img, np.array(Image.fromarray(p).resize((W, H), Image.NEAREST))

model.eval(); os.makedirs("overlays", exist_ok=True); per_img = []
for r in sel.itertuples():
    img, p = predict(r.file); h = p.shape[0]
    valid = np.zeros(p.shape, bool); valid[int(0.4*h):int(0.85*h), :] = True    # same zone as the yellow lines
    v = p[valid]; n = max(len(v), 1)
    over = img.astype(float); over[valid] = 0.5*over[valid] + 0.5*COL[p[valid]]
    out = f"overlays/{Path(r.file).stem}.jpg"; Image.fromarray(over.astype(np.uint8)).save(out, quality=80)
    per_img.append({"site": r.site, "drive": r.drive, "sol": r.sol, "lon": r.lon, "lat": r.lat, "image": out,
                    **{k: (v == i).sum() / n for i, k in enumerate(["soil_pct","bedrock_pct","sand_pct","bigrock_pct"])}})

pi = pd.DataFrame(per_img)
st = pi.groupby(["site","drive"]).agg(sol=("sol","min"), lon=("lon","first"), lat=("lat","first"),
       soil_pct=("soil_pct","mean"), bedrock_pct=("bedrock_pct","mean"),
       sand_pct=("sand_pct","mean"), bigrock_pct=("bigrock_pct","mean"), image=("image","first")).reset_index()
stops = [{"sol": int(r.sol), "lon": float(r.lon), "lat": float(r.lat),
          "soil_pct": round(r.soil_pct,3), "bedrock_pct": round(r.bedrock_pct,3),
          "sand_pct": round(r.sand_pct,3), "bigrock_pct": round(r.bigrock_pct,3),
          "image": r.image, "note": f"AI on Curiosity Navcam, sol {int(r.sol)} (site {r.site}, drive {r.drive})",
          "type": "boxwork"} for r in st.itertuples()]
json.dump(stops, open("stops.json", "w"), indent=1)
print(st[["sol","site","drive","soil_pct","bedrock_pct","sand_pct","bigrock_pct"]].round(2).to_string())

    sol  site  drive  soil_pct  bedrock_pct  sand_pct  bigrock_pct
0  4600   117   2370      0.48         0.49      0.01         0.01
1  4602   117   2490      0.01         0.99      0.00         0.00
2  4605   117   2538      0.03         0.97      0.00         0.00
3  4609   117   2730      0.00         1.00      0.00         0.00
4  4612   117   2850      0.00         1.00      0.00         0.00
5  4614   118      0      0.19         0.80      0.01         0.00
6  4616   118    162      0.29         0.54      0.18         0.00
7  4618   118    198      0.48         0.51      0.01         0.00


In [21]:
fig, ax = plt.subplots(2, 3, figsize=(18, 11))
show = pi.sort_values("sand_pct", ascending=False).head(6)
for a, (_, r) in zip(ax.flat, show.iterrows()):
    a.imshow(Image.open(r.image)); a.axis("off")
    a.set_title(f"sol {r.sol} · sand {r.sand_pct:.0%} · bedrock {r.bedrock_pct:.0%}", fontsize=11)
plt.tight_layout(); plt.show()

<Figure size 1800x1100 with 6 Axes>

In [22]:
bad = picked.loc[2, "file"]                                   # the photo with the rover arm
pi = pd.DataFrame(per_img); pi = pi[pi.image != f"overlays/{Path(bad).stem}.jpg"]
os.remove(f"overlays/{Path(bad).stem}.jpg")

st = pi.groupby(["site","drive"]).agg(sol=("sol","min"), lon=("lon","first"), lat=("lat","first"),
       soil_pct=("soil_pct","mean"), bedrock_pct=("bedrock_pct","mean"),
       sand_pct=("sand_pct","mean"), bigrock_pct=("bigrock_pct","mean"), image=("image","first")).reset_index()
stops = [{"sol": int(r.sol), "lon": float(r.lon), "lat": float(r.lat),
          "soil_pct": round(r.soil_pct,3), "bedrock_pct": round(r.bedrock_pct,3),
          "sand_pct": round(r.sand_pct,3), "bigrock_pct": round(r.bigrock_pct,3),
          "image": r.image, "note": f"AI on Curiosity Navcam, sol {int(r.sol)} (site {r.site}, drive {r.drive})",
          "type": "boxwork"} for r in st.itertuples()]
json.dump(stops, open("stops.json", "w"), indent=1)
print(len(pi), "photos ·", len(stops), "stops")
print(st[["sol","soil_pct","bedrock_pct","sand_pct","bigrock_pct"]].round(2).to_string())

19 photos · 8 stops
    sol  soil_pct  bedrock_pct  sand_pct  bigrock_pct
0  4600      0.32         0.68      0.00          0.0
1  4602      0.01         0.99      0.00          0.0
2  4605      0.03         0.97      0.00          0.0
3  4609      0.00         1.00      0.00          0.0
4  4612      0.00         1.00      0.00          0.0
5  4614      0.19         0.80      0.01          0.0
6  4616      0.29         0.54      0.18          0.0
7  4618      0.48         0.51      0.01          0.0


In [23]:
import shutil
shutil.make_archive("firmground_stops_v1", "zip", ".", "overlays")
for f in ["stops.json", "firmground_stops_v1.zip"]:
    shutil.copy(f, "/content/drive/MyDrive/firmground/")
print("saved to Drive")


saved to Drive


In [24]:
import requests
API2 = "https://mars.nasa.gov/rss/api/?feed=raw_images&category=mars2020&feedtype=json&sol={}&num=500"
os.makedirs("perseverance", exist_ok=True); prows = []
for sol in [1205, 1210, 1215]:
    ims = requests.get(API2.format(sol), timeout=60).json().get("images", [])
    for i in ims:
        ext = i.get("extended") or {}
        dim = ext.get("dimension", "(0,0)").strip("()").split(",")
        if ("NAVCAM" in i["camera"]["instrument"] and i["sample_type"] == "Full"
                and float(ext.get("mastEl", 0)) <= -15 and int(dim[1]) >= 900):   # looking DOWN at the ground, full frame
            fn = f"perseverance/{i['imageid']}.jpg"
            if not os.path.exists(fn):
                open(fn, "wb").write(requests.get(i["image_files"]["large"], timeout=60).content)
            prows.append({"file": fn, "sol": i["sol"], "id": i["imageid"]})
pp = pd.DataFrame(prows); print(len(pp), "Perseverance ground photos")

# Perseverance Navcam is colour; our model learned on grey Curiosity photos → convert to grey first
for f in pp.file:
    Image.open(f).convert("L").save(f)

pp["score"] = [pick_score(f) for f in pp.file]
best = pp.sort_values("score", ascending=False).head(6)
fig, ax = plt.subplots(2, 3, figsize=(18, 10)); os.makedirs("overlays_perseverance", exist_ok=True)
for a, r in zip(ax.flat, best.itertuples()):
    img, p = predict(r.file); h = p.shape[0]
    valid = np.zeros(p.shape, bool); valid[int(0.4*h):int(0.85*h), :] = True
    v = p[valid]; over = img.astype(float); over[valid] = 0.5*over[valid] + 0.5*COL[p[valid]]
    out = f"overlays_perseverance/{r.id}.jpg"; Image.fromarray(over.astype(np.uint8)).save(out, quality=80)
    a.imshow(over.astype(np.uint8)); a.axis("off")
    a.set_title(f"Perseverance sol {r.sol} · sand {(v==2).mean():.0%} · bedrock {(v==1).mean():.0%}", fontsize=10)
plt.tight_layout(); plt.show()

74 Perseverance ground photos


<Figure size 1800x1000 with 6 Axes>

In [ ]:
check wahedekher


In [25]:
LBL = np.array([[150,110,60],[90,90,90],[240,200,60],[200,40,40]])
def colorize(m):
    out = np.zeros((*m.shape, 3), np.uint8)
    for k in range(4): out[m == k] = LBL[k]
    return out                                        # 255 (no label) stays black

test = ds["test_min1"]
idx = [i for i in range(len(test)) if (np.isin(np.array(test[i]["label_mask"]), [2, 3])).mean() > 0.05][:4]
fig, ax = plt.subplots(len(idx), 3, figsize=(15, 5*len(idx)))
model.eval()
for row, i in enumerate(idx):
    x, y = Mars(test.select([i]))[0]
    with torch.no_grad(): p = model(x[None].to(dev)).argmax(1)[0].cpu().numpy()
    y = y.numpy(); p_show = p.copy(); p_show[y == 255] = 255
    ax[row,0].imshow(np.array(test[i]["image"].resize((SIZE, SIZE))), cmap="gray"); ax[row,0].set_title(f"test #{i}")
    ax[row,1].imshow(colorize(y));      ax[row,1].set_title("TRUTH (NASA labels)")
    ax[row,2].imshow(colorize(p_show)); ax[row,2].set_title("OUR MODEL")
    for a in ax[row]: a.axis("off")
plt.tight_layout(); plt.show()

<Figure size 1500x2000 with 12 Axes>

In [2]:
LBL = np.array([[150,110,60],[90,90,90],[240,200,60],[200,40,40]])
def colorize(m):
    out = np.zeros((*m.shape, 3), np.uint8)
    for k in range(4): out[m == k] = LBL[k]
    return out

test = ds["test_min1"]
idx = [i for i in range(len(test)) if (np.isin(np.array(test[i]["label_mask"]), [2, 3])).mean() > 0.05][:4]
fig, ax = plt.subplots(len(idx), 3, figsize=(15, 5*len(idx)))
model.eval()
for row, i in enumerate(idx):
    x, y = Mars(test.select([i]))[0]
    with torch.no_grad(): p = model(x[None].to(dev)).argmax(1)[0].cpu().numpy()
    y = y.numpy(); p_show = p.copy(); p_show[y == 255] = 255
    ax[row,0].imshow(np.array(test[i]["image"].resize((SIZE, SIZE))), cmap="gray"); ax[row,0].set_title(f"test #{i}")
    ax[row,1].imshow(colorize(y));      ax[row,1].set_title("TRUTH (NASA labels)")
    ax[row,2].imshow(colorize(p_show)); ax[row,2].set_title("OUR MODEL")
    for a in ax[row]: a.axis("off")
plt.tight_layout(); plt.show()


NameError: name 'np' is not defined

In [3]:
!pip -q install datasets segmentation-models-pytorch
from google.colab import drive; drive.mount("/content/drive")
import os, json, torch, numpy as np, pandas as pd, matplotlib.pyplot as plt
import segmentation_models_pytorch as smp, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from PIL import Image

ds = load_dataset("hassanjbara/AI4MARS")
IMG, LAB, ROV, RNG = "image", "label_mask", "rover_mask", "range_mask"
SIZE, dev = 512, "cuda"
MEAN = torch.tensor([.485,.456,.406])[:,None,None]; STD = torch.tensor([.229,.224,.225])[:,None,None]

def one(x):
    a = np.array(x); return a[..., 0] if a.ndim == 3 else a
def prepare(row):
    img = np.array(row[IMG].convert("RGB"))
    lab = one(row[LAB]).astype(np.int64).copy()
    for k in (ROV, RNG):
        if row.get(k) is not None:
            lab[one(row[k]) > 0] = 255
    return img, lab

class Mars(Dataset):
    def __init__(s, hf, aug=False): s.hf, s.aug = hf, aug
    def __len__(s): return len(s.hf)
    def __getitem__(s, i):
        img, lab = prepare(s.hf[i])
        x = torch.from_numpy(img).permute(2,0,1).float() / 255
        x = F.interpolate(x[None], size=(SIZE,SIZE), mode="bilinear", align_corners=False)[0]
        y = torch.from_numpy(lab)[None,None].float()
        y = F.interpolate(y, size=(SIZE,SIZE), mode="nearest")[0,0].long()
        return (x - MEAN) / STD, y

model = smp.DeepLabV3Plus("resnet50", encoder_weights=None, classes=4).to(dev)
model.load_state_dict(torch.load("/content/drive/MyDrive/firmground/mars_e3.pt", map_location=dev))
model.eval()
print("restored")

Mounted at /content/drive


README.md:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

data/train-00000-of-00013.parquet: reconstructing file:   0%|          |  0.00B /  475MB            

data/train-00000-of-00013.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00013.parquet: reconstructing file:   0%|          |  0.00B /  476MB            

data/train-00001-of-00013.parquet: downloading bytes:           |  0.00B            

... (29 outputs left out)

Generating test_min1 split:   0%|          | 0/322 [00:00<?, ? examples/s]

Generating test_min2 split:   0%|          | 0/322 [00:00<?, ? examples/s]

Generating test_min3 split:   0%|          | 0/322 [00:00<?, ? examples/s]

restored


In [4]:
LBL = np.array([[150,110,60],[90,90,90],[240,200,60],[200,40,40]])
def colorize(m):
    out = np.zeros((*m.shape, 3), np.uint8)
    for k in range(4): out[m == k] = LBL[k]
    return out

test = ds["test_min1"]
idx = [i for i in range(len(test)) if (np.isin(np.array(test[i]["label_mask"]), [2, 3])).mean() > 0.05][:4]
fig, ax = plt.subplots(len(idx), 3, figsize=(15, 5*len(idx)))
model.eval()
for row, i in enumerate(idx):
    x, y = Mars(test.select([i]))[0]
    with torch.no_grad(): p = model(x[None].to(dev)).argmax(1)[0].cpu().numpy()
    y = y.numpy(); p_show = p.copy(); p_show[y == 255] = 255
    ax[row,0].imshow(np.array(test[i]["image"].resize((SIZE, SIZE))), cmap="gray"); ax[row,0].set_title(f"test #{i}")
    ax[row,1].imshow(colorize(y));      ax[row,1].set_title("TRUTH (NASA labels)")
    ax[row,2].imshow(colorize(p_show)); ax[row,2].set_title("OUR MODEL")
    for a in ax[row]: a.axis("off")
plt.tight_layout(); plt.show()

<Figure size 1500x2000 with 12 Axes>

In [5]:
def sand_stats(hf, thresholds=(0.5, 0.4, 0.3, 0.2)):
    tp = {t: 0 for t in thresholds}; fp = dict(tp); fn = dict(tp)
    with torch.no_grad():
        for x, y in DataLoader(Mars(hf), batch_size=8, num_workers=2):
            prob = torch.softmax(model(x.to(dev)), 1)[:, 2].cpu().numpy()   # probability of SAND
            y = y.numpy(); m = y != 255; truth = (y == 2) & m
            for t in thresholds:
                pred = (prob >= t) & m
                tp[t] += (pred & truth).sum(); fp[t] += (pred & ~truth).sum(); fn[t] += (~pred & truth).sum()
    for t in thresholds:
        rec = tp[t] / (tp[t] + fn[t]); prec = tp[t] / max(tp[t] + fp[t], 1); iou = tp[t] / (tp[t] + fp[t] + fn[t])
        print(f"sand if prob ≥ {t}:  found {rec:.0%} of real sand (recall) · {prec:.0%} of 'sand' is real (precision) · IoU {iou:.2f}")

sand_stats(ds["test_min1"])

sand if prob ≥ 0.5:  found 90% of real sand (recall) · 96% of 'sand' is real (precision) · IoU 0.86
sand if prob ≥ 0.4:  found 91% of real sand (recall) · 94% of 'sand' is real (precision) · IoU 0.86
sand if prob ≥ 0.3:  found 92% of real sand (recall) · 93% of 'sand' is real (precision) · IoU 0.86
sand if prob ≥ 0.2:  found 93% of real sand (recall) · 91% of 'sand' is real (precision) · IoU 0.85


In [6]:
LBL = np.array([[150,110,60],[90,90,90],[240,200,60],[200,40,40]])
def colorize(m):
    out = np.zeros((*m.shape, 3), np.uint8)
    for k in range(4): out[m == k] = LBL[k]
    return out

test = ds["test_min1"]
idx = [i for i in range(len(test)) if (np.isin(np.array(test[i]["label_mask"]), [2, 3])).mean() > 0.05][:4]
fig, ax = plt.subplots(len(idx), 3, figsize=(15, 5*len(idx)))
model.eval()
for row, i in enumerate(idx):
    x, y = Mars(test.select([i]))[0]
    with torch.no_grad(): p = model(x[None].to(dev)).argmax(1)[0].cpu().numpy()
    y = y.numpy(); p_show = p.copy(); p_show[y == 255] = 255
    ax[row,0].imshow(np.array(test[i]["image"].resize((SIZE, SIZE))), cmap="gray"); ax[row,0].set_title(f"test #{i}")
    ax[row,1].imshow(colorize(y));      ax[row,1].set_title("TRUTH (NASA labels)")
    ax[row,2].imshow(colorize(p_show)); ax[row,2].set_title("OUR MODEL")
    for a in ax[row]: a.axis("off")
plt.tight_layout(); plt.show()

<Figure size 1500x2000 with 12 Axes>

In [7]:
def sand_stats(hf, thresholds=(0.5, 0.4, 0.3, 0.2)):
    tp = {t: 0 for t in thresholds}; fp = dict(tp); fn = dict(tp)
    with torch.no_grad():
        for x, y in DataLoader(Mars(hf), batch_size=8, num_workers=2):
            prob = torch.softmax(model(x.to(dev)), 1)[:, 2].cpu().numpy()   # probability of SAND
            y = y.numpy(); m = y != 255; truth = (y == 2) & m
            for t in thresholds:
                pred = (prob >= t) & m
                tp[t] += (pred & truth).sum(); fp[t] += (pred & ~truth).sum(); fn[t] += (~pred & truth).sum()
    for t in thresholds:
        rec = tp[t] / (tp[t] + fn[t]); prec = tp[t] / max(tp[t] + fp[t], 1); iou = tp[t] / (tp[t] + fp[t] + fn[t])
        print(f"sand if prob ≥ {t}:  found {rec:.0%} of real sand (recall) · {prec:.0%} of 'sand' is real (precision) · IoU {iou:.2f}")

sand_stats(ds["test_min1"])

sand if prob ≥ 0.5:  found 90% of real sand (recall) · 96% of 'sand' is real (precision) · IoU 0.86
sand if prob ≥ 0.4:  found 91% of real sand (recall) · 94% of 'sand' is real (precision) · IoU 0.86
sand if prob ≥ 0.3:  found 92% of real sand (recall) · 93% of 'sand' is real (precision) · IoU 0.86
sand if prob ≥ 0.2:  found 93% of real sand (recall) · 91% of 'sand' is real (precision) · IoU 0.85


In [8]:
import requests, time
BASE = "/content/drive/MyDrive/firmground"
os.makedirs(f"{BASE}/navcam", exist_ok=True)

wp = requests.get("https://mars.nasa.gov/mmgis-maps/MSL/Layers/json/MSL_waypoints.json").json()
df = pd.DataFrame([f["properties"] for f in wp["features"]])

API = "https://mars.nasa.gov/api/v1/raw_image_items/"
def navcam(sol, cam):
    q = {"order": "sol desc", "per_page": 100, "page": 0, "condition_1": "msl:mission",
         "condition_2": f"{sol}:sol:in", "condition_3": f"{cam}:instrument"}
    items = requests.get(API, params=q, timeout=30).json().get("items", [])
    return [i for i in items if not i["is_thumbnail"] and (i.get("extended") or {}).get("sample_type") == "full"]

CAP = 12                                   # max photos per stop (site, drive)
count, rows = {}, []
for sol in range(4534, 4806):
    for cam in ("NAV_RIGHT_B", "NAV_LEFT_B"):
        try:
            for it in navcam(sol, cam):
                key = (it["site"], it["drive"])
                if count.get(key, 0) >= CAP: continue
                fn = f"{BASE}/navcam/{it['imageid']}.jpg"
                if not os.path.exists(fn):
                    open(fn, "wb").write(requests.get(it["https_url"], timeout=60).content)
                count[key] = count.get(key, 0) + 1
                rows.append({"file": fn, "sol": it["sol"], "site": it["site"], "drive": it["drive"]})
        except Exception as e:
            print(sol, cam, "error:", e)
        time.sleep(0.3)
    if sol % 25 == 0: print("sol", sol, "·", len(rows), "photos so far")
photos = pd.DataFrame(rows); photos.to_csv(f"{BASE}/photos_all.csv", index=False)
print("DONE:", len(photos), "photos ·", len(count), "stops")

sol 4550 · 121 photos so far
sol 4575 · 230 photos so far
sol 4600 · 420 photos so far
sol 4625 · 577 photos so far
sol 4650 · 729 photos so far
sol 4675 · 866 photos so far
sol 4700 · 934 photos so far
sol 4725 · 967 photos so far
sol 4750 · 995 photos so far
sol 4775 · 1011 photos so far
sol 4800 · 1023 photos so far
DONE: 1039 photos · 131 stops


In [11]:
photos = photos[[Image.open(f).size == (1024, 1024) for f in photos.file]]
geo = photos.merge(df[["site","drive","lon","lat"]], on=["site","drive"], how="inner").reset_index(drop=True)

def pick_score(f):
    a = np.array(Image.open(f).convert("L"), float); h = a.shape[0]
    top, band = a[: int(0.2*h)], a[int(0.4*h): int(0.85*h)]
    return min(top.std()/25, 1) + min(band.std()/30, 1) - 3*(band < 35).mean()

geo["score"] = [pick_score(f) for f in geo.file]
picked = (geo[geo.score > 1.2].sort_values("score", ascending=False)
             .groupby(["site","drive"]).head(3).sort_values(["sol","site","drive"]).reset_index(drop=True))
picked.to_csv(f"{BASE}/picked.csv")
print(len(picked), "photos for", picked.groupby(["site","drive"]).ngroups, "stops")

def show(start, n=60):
    sub = picked.iloc[start:start+n]
    rows = (len(sub) + 9) // 10
    fig, ax = plt.subplots(rows, 10, figsize=(30, 3.2*rows))
    for a in ax.flat: a.axis("off")
    for a, (i, r) in zip(ax.flat, sub.iterrows()):
        im = np.array(Image.open(r.file).convert("L")); h = im.shape[0]
        a.imshow(im, cmap="gray"); a.axhline(0.4*h, color="yellow", lw=1); a.axhline(0.85*h, color="yellow", lw=1)
        a.set_title(f"#{i} · {r.sol}", fontsize=10)
    plt.tight_layout(); plt.show()
show(0)


198 photos for 66 stops


<Figure size 3000x1920 with 60 Axes>

In [12]:
photos = photos[[Image.open(f).size == (1024, 1024) for f in photos.file]]
geo = photos.merge(df[["site","drive","lon","lat"]], on=["site","drive"], how="inner").reset_index(drop=True)

def pick_score(f):
    a = np.array(Image.open(f).convert("L"), float); h = a.shape[0]
    top, band = a[: int(0.2*h)], a[int(0.4*h): int(0.85*h)]
    return min(top.std()/25, 1) + min(band.std()/30, 1) - 3*(band < 35).mean()

geo["score"] = [pick_score(f) for f in geo.file]
picked = (geo[geo.score > 1.2].sort_values("score", ascending=False)
             .groupby(["site","drive"]).head(3).sort_values(["sol","site","drive"]).reset_index(drop=True))
picked.to_csv(f"{BASE}/picked.csv")
print(len(picked), "photos for", picked.groupby(["site","drive"]).ngroups, "stops")

def show(start, n=30):
    sub = picked.iloc[start:start+n]
    rows = (len(sub) + 5) // 6
    fig, ax = plt.subplots(rows, 6, figsize=(24, 4.3*rows))
    for a in ax.flat: a.axis("off")
    for a, (i, r) in zip(ax.flat, sub.iterrows()):
        im = np.array(Image.open(r.file).convert("L")); h = im.shape[0]
        a.imshow(im, cmap="gray"); a.axhline(0.4*h, color="yellow", lw=2); a.axhline(0.85*h, color="yellow", lw=2)
        a.set_title(f"#{i} · sol {r.sol}", fontsize=16, fontweight="bold")
    plt.tight_layout(); plt.show()
show(0)

198 photos for 66 stops


<Figure size 2400x2150 with 30 Axes>

In [13]:
photos = photos[[Image.open(f).size == (1024, 1024) for f in photos.file]]
geo = photos.merge(df[["site","drive","lon","lat"]], on=["site","drive"], how="inner").reset_index(drop=True)

def pick_score(f):
    a = np.array(Image.open(f).convert("L"), float); h = a.shape[0]
    top, band = a[: int(0.2*h)], a[int(0.4*h): int(0.85*h)]
    return min(top.std()/25, 1) + min(band.std()/30, 1) - 3*(band < 35).mean()

geo["score"] = [pick_score(f) for f in geo.file]
picked = (geo[geo.score > 1.2].sort_values("score", ascending=False)
             .groupby(["site","drive"]).head(3).sort_values(["sol","site","drive"]).reset_index(drop=True))
picked.to_csv(f"{BASE}/picked.csv")
print(len(picked), "photos for", picked.groupby(["site","drive"]).ngroups, "stops")

def show(start, n=60):
    sub = picked.iloc[start:start+n]
    rows = (len(sub) + 5) // 6
    fig, ax = plt.subplots(rows, 6, figsize=(24, 4.3*rows))
    for a in ax.flat: a.axis("off")
    for a, (i, r) in zip(ax.flat, sub.iterrows()):
        im = np.array(Image.open(r.file).convert("L")); h = im.shape[0]
        a.imshow(im, cmap="gray"); a.axhline(0.4*h, color="yellow", lw=2); a.axhline(0.85*h, color="yellow", lw=2)
        a.set_title(f"#{i} · sol {r.sol}", fontsize=16, fontweight="bold")
    plt.tight_layout(); plt.show()
show(0)

198 photos for 66 stops


<Figure size 2400x4300 with 60 Axes>

In [14]:
photos = photos[[Image.open(f).size == (1024, 1024) for f in photos.file]]
geo = photos.merge(df[["site","drive","lon","lat"]], on=["site","drive"], how="inner").reset_index(drop=True)

def pick_score(f):
    a = np.array(Image.open(f).convert("L"), float); h = a.shape[0]
    top, band = a[: int(0.2*h)], a[int(0.4*h): int(0.85*h)]
    return min(top.std()/25, 1) + min(band.std()/30, 1) - 3*(band < 35).mean()

geo["score"] = [pick_score(f) for f in geo.file]
picked = (geo[geo.score > 1.2].sort_values("score", ascending=False)
             .groupby(["site","drive"]).head(3).sort_values(["sol","site","drive"]).reset_index(drop=True))
picked.to_csv(f"{BASE}/picked.csv")
print(len(picked), "photos for", picked.groupby(["site","drive"]).ngroups, "stops")

def show(start, n=90):
    sub = picked.iloc[start:start+n]
    rows = (len(sub) + 5) // 6
    fig, ax = plt.subplots(rows, 6, figsize=(24, 4.3*rows))
    for a in ax.flat: a.axis("off")
    for a, (i, r) in zip(ax.flat, sub.iterrows()):
        im = np.array(Image.open(r.file).convert("L")); h = im.shape[0]
        a.imshow(im, cmap="gray"); a.axhline(0.4*h, color="yellow", lw=2); a.axhline(0.85*h, color="yellow", lw=2)
        a.set_title(f"#{i} · sol {r.sol}", fontsize=16, fontweight="bold")
    plt.tight_layout(); plt.show()
show(0)

198 photos for 66 stops


<Figure size 2400x6450 with 90 Axes>

In [15]:
photos = photos[[Image.open(f).size == (1024, 1024) for f in photos.file]]
geo = photos.merge(df[["site","drive","lon","lat"]], on=["site","drive"], how="inner").reset_index(drop=True)

def pick_score(f):
    a = np.array(Image.open(f).convert("L"), float); h = a.shape[0]
    top, band = a[: int(0.2*h)], a[int(0.4*h): int(0.85*h)]
    return min(top.std()/25, 1) + min(band.std()/30, 1) - 3*(band < 35).mean()

geo["score"] = [pick_score(f) for f in geo.file]
picked = (geo[geo.score > 1.2].sort_values("score", ascending=False)
             .groupby(["site","drive"]).head(3).sort_values(["sol","site","drive"]).reset_index(drop=True))
picked.to_csv(f"{BASE}/picked.csv")
print(len(picked), "photos for", picked.groupby(["site","drive"]).ngroups, "stops")

def show(start, n=120):
    sub = picked.iloc[start:start+n]
    rows = (len(sub) + 5) // 6
    fig, ax = plt.subplots(rows, 6, figsize=(24, 4.3*rows))
    for a in ax.flat: a.axis("off")
    for a, (i, r) in zip(ax.flat, sub.iterrows()):
        im = np.array(Image.open(r.file).convert("L")); h = im.shape[0]
        a.imshow(im, cmap="gray"); a.axhline(0.4*h, color="yellow", lw=2); a.axhline(0.85*h, color="yellow", lw=2)
        a.set_title(f"#{i} · sol {r.sol}", fontsize=16, fontweight="bold")
    plt.tight_layout(); plt.show()
show(0)

198 photos for 66 stops


<Figure size 2400x8600 with 120 Axes>

In [ ]:
photos = photos[[Image.open(f).size == (1024, 1024) for f in photos.file]]
geo = photos.merge(df[["site","drive","lon","lat"]], on=["site","drive"], how="inner").reset_index(drop=True)

def pick_score(f):
    a = np.array(Image.open(f).convert("L"), float); h = a.shape[0]
    top, band = a[: int(0.2*h)], a[int(0.4*h): int(0.85*h)]
    return min(top.std()/25, 1) + min(band.std()/30, 1) - 3*(band < 35).mean()

geo["score"] = [pick_score(f) for f in geo.file]
picked = (geo[geo.score > 1.2].sort_values("score", ascending=False)
             .groupby(["site","drive"]).head(3).sort_values(["sol","site","drive"]).reset_index(drop=True))
picked.to_csv(f"{BASE}/picked.csv")
print(len(picked), "photos for", picked.groupby(["site","drive"]).ngroups, "stops")

def show(start, n=150):
    sub = picked.iloc[start:start+n]
    rows = (len(sub) + 5) // 6
    fig, ax = plt.subplots(rows, 6, figsize=(24, 4.3*rows))
    for a in ax.flat: a.axis("off")
    for a, (i, r) in zip(ax.flat, sub.iterrows()):
        im = np.array(Image.open(r.file).convert("L")); h = im.shape[0]
        a.imshow(im, cmap="gray"); a.axhline(0.4*h, color="yellow", lw=2); a.axhline(0.85*h, color="yellow", lw=2)
        a.set_title(f"#{i} · sol {r.sol}", fontsize=16, fontweight="bold")
    plt.tight_layout(); plt.show()
show(0)

In [ ]:
photos = photos[[Image.open(f).size == (1024, 1024) for f in photos.file]]
geo = photos.merge(df[["site","drive","lon","lat"]], on=["site","drive"], how="inner").reset_index(drop=True)

def pick_score(f):
    a = np.array(Image.open(f).convert("L"), float); h = a.shape[0]
    top, band = a[: int(0.2*h)], a[int(0.4*h): int(0.85*h)]
    return min(top.std()/25, 1) + min(band.std()/30, 1) - 3*(band < 35).mean()

geo["score"] = [pick_score(f) for f in geo.file]
picked = (geo[geo.score > 1.2].sort_values("score", ascending=False)
             .groupby(["site","drive"]).head(3).sort_values(["sol","site","drive"]).reset_index(drop=True))
picked.to_csv(f"{BASE}/picked.csv")
print(len(picked), "photos for", picked.groupby(["site","drive"]).ngroups, "stops")

def show(start, n=180):
    sub = picked.iloc[start:start+n]
    rows = (len(sub) + 5) // 6
    fig, ax = plt.subplots(rows, 6, figsize=(24, 4.3*rows))
    for a in ax.flat: a.axis("off")
    for a, (i, r) in zip(ax.flat, sub.iterrows()):
        im = np.array(Image.open(r.file).convert("L")); h = im.shape[0]
        a.imshow(im, cmap="gray"); a.axhline(0.4*h, color="yellow", lw=2); a.axhline(0.85*h, color="yellow", lw=2)
        a.set_title(f"#{i} · sol {r.sol}", fontsize=16, fontweight="bold")
    plt.tight_layout(); plt.show()
show(0)

In [16]:
photos = photos[[Image.open(f).size == (1024, 1024) for f in photos.file]]
geo = photos.merge(df[["site","drive","lon","lat"]], on=["site","drive"], how="inner").reset_index(drop=True)

def pick_score(f):
    a = np.array(Image.open(f).convert("L"), float); h = a.shape[0]
    top, band = a[: int(0.2*h)], a[int(0.4*h): int(0.85*h)]
    return min(top.std()/25, 1) + min(band.std()/30, 1) - 3*(band < 35).mean()

geo["score"] = [pick_score(f) for f in geo.file]
picked = (geo[geo.score > 1.2].sort_values("score", ascending=False)
             .groupby(["site","drive"]).head(3).sort_values(["sol","site","drive"]).reset_index(drop=True))
picked.to_csv(f"{BASE}/picked.csv")
print(len(picked), "photos for", picked.groupby(["site","drive"]).ngroups, "stops")

def show(start, n=59):
    sub = picked.iloc[start:start+n]
    rows = (len(sub) + 5) // 6
    fig, ax = plt.subplots(rows, 6, figsize=(24, 4.3*rows))
    for a in ax.flat: a.axis("off")
    for a, (i, r) in zip(ax.flat, sub.iterrows()):
        im = np.array(Image.open(r.file).convert("L")); h = im.shape[0]
        a.imshow(im, cmap="gray"); a.axhline(0.4*h, color="yellow", lw=2); a.axhline(0.85*h, color="yellow", lw=2)
        a.set_title(f"#{i} · sol {r.sol}", fontsize=16, fontweight="bold")
    plt.tight_layout(); plt.show()
show(0)

198 photos for 66 stops


<Figure size 2400x4300 with 60 Axes>

In [17]:
photos = photos[[Image.open(f).size == (1024, 1024) for f in photos.file]]
geo = photos.merge(df[["site","drive","lon","lat"]], on=["site","drive"], how="inner").reset_index(drop=True)

def pick_score(f):
    a = np.array(Image.open(f).convert("L"), float); h = a.shape[0]
    top, band = a[: int(0.2*h)], a[int(0.4*h): int(0.85*h)]
    return min(top.std()/25, 1) + min(band.std()/30, 1) - 3*(band < 35).mean()

geo["score"] = [pick_score(f) for f in geo.file]
picked = (geo[geo.score > 1.2].sort_values("score", ascending=False)
             .groupby(["site","drive"]).head(3).sort_values(["sol","site","drive"]).reset_index(drop=True))
picked.to_csv(f"{BASE}/picked.csv")
print(len(picked), "photos for", picked.groupby(["site","drive"]).ngroups, "stops")

def show(start, n=119):
    sub = picked.iloc[start:start+n]
    rows = (len(sub) + 5) // 6
    fig, ax = plt.subplots(rows, 6, figsize=(24, 4.3*rows))
    for a in ax.flat: a.axis("off")
    for a, (i, r) in zip(ax.flat, sub.iterrows()):
        im = np.array(Image.open(r.file).convert("L")); h = im.shape[0]
        a.imshow(im, cmap="gray"); a.axhline(0.4*h, color="yellow", lw=2); a.axhline(0.85*h, color="yellow", lw=2)
        a.set_title(f"#{i} · sol {r.sol}", fontsize=16, fontweight="bold")
    plt.tight_layout(); plt.show()
show(0)

198 photos for 66 stops


<Figure size 2400x8600 with 120 Axes>

In [18]:
show(119, 80)

<Figure size 2400x6020 with 84 Axes>

In [19]:
remove = [0, 1, 2, 9, 12, 13, 14, 16, 18, 19, 20, 25, 26, 28, 29, 35, 46, 49, 52, 55, 56, 58,
          59, 60, 61, 63, 64, 65, 66, 67, 69, 70, 71, 76, 77, 78, 79, 80, 85, 100, 101, 106, 107, 112, 113, 116,
          121, 129, 130, 131, 137, 143, 150, 151, 152, 153, 156, 157, 158, 162, 163, 164, 166, 167,
          174, 175, 177, 179, 182, 184, 188, 192, 194]
sel = picked.drop(index=remove).reset_index(drop=True)
print("photos kept:", len(sel))

COL = np.array([[150,110,60],[90,90,90],[240,200,60],[200,40,40]])   # soil, bedrock, sand, big rock
SAND_T = 0.30                                                        # safety rule: sand if model is >=30% sure
def predict(path):
    img = np.array(Image.open(path).convert("RGB")); H, W = img.shape[:2]
    x = torch.from_numpy(img).permute(2,0,1).float() / 255
    x = F.interpolate(x[None], size=(SIZE,SIZE), mode="bilinear", align_corners=False)[0]
    with torch.no_grad():
        prob = torch.softmax(model(((x - MEAN) / STD)[None].to(dev)), 1)[0]
    p = prob.argmax(0); p[prob[2] >= SAND_T] = 2
    return img, np.array(Image.fromarray(p.cpu().numpy().astype(np.uint8)).resize((W, H), Image.NEAREST))

model.eval(); os.makedirs(f"{BASE}/overlays_v2", exist_ok=True); per_img = []
for r in sel.itertuples():
    img, p = predict(r.file); h = p.shape[0]
    valid = np.zeros(p.shape, bool); valid[int(0.4*h):int(0.85*h), :] = True
    v = p[valid]; n = max(len(v), 1)
    over = img.astype(float); over[valid] = 0.5*over[valid] + 0.5*COL[p[valid]]
    name = os.path.basename(r.file)
    Image.fromarray(over.astype(np.uint8)).save(f"{BASE}/overlays_v2/{name}", quality=80)
    per_img.append({"site": r.site, "drive": r.drive, "sol": r.sol, "lon": r.lon, "lat": r.lat,
                    "image": f"overlays/{name}", "file": r.file,
                    **{k: (v == i).sum() / n for i, k in enumerate(["soil_pct","bedrock_pct","sand_pct","bigrock_pct"])}})

pi = pd.DataFrame(per_img); pi.to_csv(f"{BASE}/per_image_v2.csv", index=False)
st = pi.groupby(["site","drive"]).agg(sol=("sol","min"), lon=("lon","first"), lat=("lat","first"),
       soil_pct=("soil_pct","mean"), bedrock_pct=("bedrock_pct","mean"), sand_pct=("sand_pct","mean"),
       bigrock_pct=("bigrock_pct","mean"), image=("image","first")).reset_index()
stops = [{"sol": int(r.sol), "lon": float(r.lon), "lat": float(r.lat),
          "soil_pct": round(r.soil_pct,3), "bedrock_pct": round(r.bedrock_pct,3),
          "sand_pct": round(r.sand_pct,3), "bigrock_pct": round(r.bigrock_pct,3), "image": r.image,
          "note": f"AI on Curiosity Navcam, sol {int(r.sol)} (site {r.site}, drive {r.drive}), sand threshold 0.3",
          "type": "boxwork" if r.sol >= 4600 else "stop"} for r in st.itertuples()]
json.dump(stops, open(f"{BASE}/stops_v2.json", "w"), indent=1)

print(len(stops), "stops saved to Drive (stops_v2.json + overlays_v2/)")
print("\nTop 5 sandiest stops:")
print(st.sort_values("sand_pct", ascending=False).head(5)[["sol","sand_pct","bedrock_pct","soil_pct","bigrock_pct"]].round(2).to_string())

photos kept: 125
56 stops saved to Drive (stops_v2.json + overlays_v2/)

Top 5 sandiest stops:
     sol  sand_pct  bedrock_pct  soil_pct  bigrock_pct
14  4582      0.29         0.65      0.05         0.01
25  4616      0.14         0.62      0.24         0.00
3   4544      0.13         0.85      0.01         0.01
51  4741      0.12         0.11      0.78         0.00
55  4805      0.02         0.98      0.00         0.00


In [20]:
import shutil
# 1) colour count with our rule
def colour(r):
    if r.sand_pct > 0.30 or r.bigrock_pct > 0.02: return "red"
    if r.sand_pct >= 0.10: return "orange"
    return "green"
st["colour"] = st.apply(colour, axis=1); print(st.colour.value_counts().to_string())

# 2) look at the sandiest photos
show_df = pi.sort_values("sand_pct", ascending=False).head(6)
fig, ax = plt.subplots(2, 3, figsize=(18, 11))
for a, (_, r) in zip(ax.flat, show_df.iterrows()):
    a.imshow(Image.open(f"{BASE}/overlays_v2/{os.path.basename(r.file)}")); a.axis("off")
    a.set_title(f"sol {r.sol} · sand {r.sand_pct:.0%} · bedrock {r.bedrock_pct:.0%}", fontsize=12)
plt.tight_layout(); plt.show()

# 3) one zip for the team: stops.json + overlays/
os.makedirs("/content/v2/overlays", exist_ok=True)
shutil.copy(f"{BASE}/stops_v2.json", "/content/v2/stops.json")
for f in os.listdir(f"{BASE}/overlays_v2"): shutil.copy(f"{BASE}/overlays_v2/{f}", "/content/v2/overlays/")
shutil.make_archive(f"{BASE}/firmground_stops_v2", "zip", "/content/v2")
print("zip ready in Drive: firmground_stops_v2.zip")

colour
green     49
orange     4
red        3


<Figure size 1800x1100 with 6 Axes>

zip ready in Drive: firmground_stops_v2.zip


In [21]:
red = st[st.colour == "red"]
print(red[["sol","sand_pct","bigrock_pct","bedrock_pct"]].round(3).to_string())
ph = pi[pi.set_index(["site","drive"]).index.isin(red.set_index(["site","drive"]).index)]
fig, ax = plt.subplots(1, len(ph), figsize=(6*len(ph), 6))
for a, (_, r) in zip(np.atleast_1d(ax), ph.iterrows()):
    a.imshow(Image.open(f"{BASE}/overlays_v2/{os.path.basename(r.file)}")); a.axis("off")
    a.set_title(f"sol {r.sol} · bigrock {r.bigrock_pct:.1%} · sand {r.sand_pct:.0%}", fontsize=12)
plt.tight_layout(); plt.show()

     sol  sand_pct  bigrock_pct  bedrock_pct
5   4552     0.000        0.027        0.825
13  4580     0.000        0.091        0.086
24  4614     0.005        0.032        0.663


<Figure size 3600x600 with 6 Axes>

In [22]:
bad_files = picked.loc[[24, 92], "file"].tolist()         # 4552 shadow, 4614 far view
pi2 = pi[~pi.file.isin(bad_files)]
st = pi2.groupby(["site","drive"]).agg(sol=("sol","min"), lon=("lon","first"), lat=("lat","first"),
       soil_pct=("soil_pct","mean"), bedrock_pct=("bedrock_pct","mean"), sand_pct=("sand_pct","mean"),
       bigrock_pct=("bigrock_pct","mean"), image=("image","first")).reset_index()
stops = [{"sol": int(r.sol), "lon": float(r.lon), "lat": float(r.lat),
          "soil_pct": round(r.soil_pct,3), "bedrock_pct": round(r.bedrock_pct,3),
          "sand_pct": round(r.sand_pct,3), "bigrock_pct": round(r.bigrock_pct,3), "image": r.image,
          "note": f"AI on Curiosity Navcam, sol {int(r.sol)} (site {r.site}, drive {r.drive}), sand threshold 0.3",
          "type": "boxwork" if r.sol >= 4600 else "stop"} for r in st.itertuples()]
json.dump(stops, open(f"{BASE}/stops_v2.json", "w"), indent=1)
st["colour"] = st.apply(colour, axis=1); print(len(stops), "stops"); print(st.colour.value_counts().to_string())

shutil.copy(f"{BASE}/stops_v2.json", "/content/v2/stops.json")
shutil.make_archive(f"{BASE}/firmground_stops_v2", "zip", "/content/v2")
print("zip updated in Drive")

55 stops
colour
green     50
orange     4
red        1
zip updated in Drive
